# 08 - Consistency Test (LLM Output Reliability)

Before scaling experiments, we verify the LLM produces stable output.
Same student, same problems, same prompts — run 3 times each and measure
tag agreement across runs.

If output is consistent, single-run results are trustworthy.
If not, we need multiple runs with majority voting.

In [3]:
import json
import time
import numpy as np
from pathlib import Path
import pandas as pd
from google.genai import types

# Same ROOT / sys.path setup as notebook 05
ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.experiment_utils import create_client, load_best_attempts_df
from lib.llm_batch_analyzer import format_submissions, clean_json_response
from lib.mental_model import load_skill_map, calculate_student_profile, build_prerequisite_graph, get_weak_skills, build_mental_model_payload
try:
    from lib.prompt_strategies import build_curriculum_aware_prompt, KC_TAGS
except ModuleNotFoundError:
    from lib.prompts import build_curriculum_aware_prompt, KC_TAGS
from utils.dataset import load_topics_json, load_problem_descriptions

MODEL_ID = 'gemini-2.5-flash'
TARGET_STUDENT_ID = 14359
NUM_RUNS = 3
SLEEP_SECONDS = 1.0

client = create_client()
print(f'Ready. Model={MODEL_ID}, Student={TARGET_STUDENT_ID}, Runs per problem={NUM_RUNS}')

Ready. Model=gemini-2.5-flash, Student=14359, Runs per problem=3


In [4]:
best_attempts_df = load_best_attempts_df()
student_df = best_attempts_df[best_attempts_df['SubjectID'] == TARGET_STUDENT_ID].copy()
student_df = student_df.drop_duplicates(subset=['ProblemID']).copy()

score_max = float(student_df['Score'].max())
if score_max <= 1.0:
    student_df['ScorePct'] = student_df['Score'] * 100.0
else:
    student_df['ScorePct'] = student_df['Score']

# Select ONLY the failing problems we used in experiment 05
FAILING_PROBLEM_IDS = [108, 32, 107, 40, 34]
test_df = student_df[student_df['ProblemID'].isin(FAILING_PROBLEM_IDS)].copy()
test_df = test_df.sort_values('ScorePct', ascending=False).reset_index(drop=True)

print(f"Test problems: {len(test_df)}")
display(test_df[['ProblemID', 'Score', 'ScorePct']])

# Build mental model
skill_map, all_skills = load_skill_map()
student_profile = calculate_student_profile(TARGET_STUDENT_ID, best_attempts_df, skill_map, all_skills)
weak_skills = get_weak_skills(student_profile, threshold=0.6)
G = build_prerequisite_graph()
mental_model = build_mental_model_payload(
    student_id=TARGET_STUDENT_ID, profile=student_profile,
    weak_skill_pairs=weak_skills, graph=G,
)

# Build prompts
topics = load_topics_json() or {}
problem_descriptions = load_problem_descriptions() or {}
selected_problem_ids = [int(pid) for pid in test_df['ProblemID'].tolist()]

baseline_prompt = build_curriculum_aware_prompt(
    topics=topics, problems=problem_descriptions, focus_problem_ids=selected_problem_ids,
)

enriched_prompt = baseline_prompt + "\n\nAdditional Student Mental Model Context:\n" + json.dumps(mental_model, indent=2) + "\n\nUse this context to better judge likely misconceptions and future risks."

weak_skill_names = [s[0] for s in weak_skills]
print(f"Weak skills: {weak_skill_names}")
print(f"Baseline prompt: {len(baseline_prompt)} chars")
print(f"Enriched prompt: {len(enriched_prompt)} chars")

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Test problems: 5


,ProblemID,Score,ScorePct
0,108,0.684211,68.4211
1,32,0.545455,54.5455
2,107,0.545455,54.5455
3,40,0.384615,38.4615
4,34,0.142857,14.2857


Weak skills: ['StringConcat', 'While']
Baseline prompt: 12703 chars
Enriched prompt: 13704 chars


In [5]:
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KC_SET = set(EXACT_KC_TAGS)

if set(KC_TAGS) != VALID_KC_SET:
    print('Warning: imported KC_TAGS differs from the required exact KC vocabulary. Validation will use EXACT_KC_TAGS.')

prompts_df = pd.read_csv(ROOT / 'dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv')

def add_mental_model_context(base_prompt: str, mental_model_payload: dict) -> str:
    context = json.dumps(mental_model_payload, indent=2)
    return (
        base_prompt
        + "\n\nAdditional Student Mental Model Context:\n"
        + context
        + "\n\nUse this context to better judge likely misconceptions and future risks."
    )

def run_one_call(submission_row, system_instruction: str):
    formatted_input = format_submissions([submission_row.to_dict()])
    t0 = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
                response_mime_type='application/json',
            ),
        )
        raw_text = response.text if response and response.text else '{}'
        parsed = json.loads(clean_json_response(raw_text))
        return parsed, round(time.time() - t0, 3), None
    except Exception as e:
        return None, round(time.time() - t0, 3), str(e)

def extract_expected_kc_tags(problem_id: int) -> list[str]:
    row = prompts_df[prompts_df['ProblemID'] == int(problem_id)]
    if row.empty:
        return []
    row = row.iloc[0]
    tags = []
    for tag in EXACT_KC_TAGS:
        value = row.get(tag, 0)
        if pd.notna(value) and float(value) == 1.0:
            tags.append(tag)
    return tags

def extract_kc_tags(output_obj) -> tuple[list[str], list[str]]:
    """Extract KC tags from Curriculum-Aware LLM output structure."""
    if not isinstance(output_obj, dict):
        return [], []

    valid_tags = set()
    invalid_tags = set()
    analysis_list = output_obj.get("student_analysis", [])
    if not isinstance(analysis_list, list):
        analysis_list = []

    for analysis in analysis_list:
        if not isinstance(analysis, dict): continue
        for gap in analysis.get("knowledge_gaps", []):
            tag = ""
            if isinstance(gap, dict):
                tag = gap.get("missing_concept", "")
            elif isinstance(gap, str):
                tag = gap
            tag = tag.strip()
            if tag:
                if tag in VALID_KC_SET:
                    valid_tags.add(tag)
                else:
                    invalid_tags.add(tag)

        for pred in analysis.get("future_predictions", []):
            tag = ""
            if isinstance(pred, dict):
                tag = pred.get("at_risk_topic", "")
            elif isinstance(pred, str):
                tag = pred
            tag = tag.strip()
            if tag:
                if tag in VALID_KC_SET:
                    valid_tags.add(tag)
                else:
                    invalid_tags.add(tag)

    for key in ["knowledge_gaps", "future_predictions"]:
        val = output_obj.get(key, [])
        if isinstance(val, list):
            for item in val:
                if isinstance(item, str):
                    item = item.strip()
                    if item in VALID_KC_SET:
                        valid_tags.add(item)
                    elif item:
                        invalid_tags.add(item)

    return sorted(valid_tags), sorted(invalid_tags)

enriched_prompt = add_mental_model_context(baseline_prompt, mental_model)
print('Helpers ready. Enriched prompt prepared.')
# Then add this:
def jaccard_similarity(set_a: set, set_b: set) -> float:
    """Jaccard similarity between two sets. 1.0 = identical, 0.0 = no overlap."""
    if not set_a and not set_b:
        return 1.0  # Both empty = identical
    union = set_a | set_b
    if not union:
        return 1.0
    return len(set_a & set_b) / len(union)

print("Helpers ready.")

Helpers ready. Enriched prompt prepared.
Helpers ready.


In [6]:
results = []

for _, sub in test_df.iterrows():
    pid = int(sub['ProblemID'])
    score_pct = float(sub['ScorePct'])
    print(f"\n--- Problem {pid} (Score: {score_pct:.1f}%) ---")

    for condition_name, prompt in [("Baseline", baseline_prompt), ("Enriched", enriched_prompt)]:
        run_tag_sets = []

        for run_num in range(1, NUM_RUNS + 1):
            out, t, err = run_one_call(sub, prompt)
            tags, invalid = extract_kc_tags(out if err is None else {})
            run_tag_sets.append(set(tags))
            print(f"  {condition_name} Run {run_num}: {sorted(tags)} ({t}s)")
            if invalid:
                print(f"    Invalid: {invalid}")
            time.sleep(SLEEP_SECONDS)

        # Calculate pairwise Jaccard similarity
        pairs = []
        for i in range(NUM_RUNS):
            for j in range(i + 1, NUM_RUNS):
                j_score = jaccard_similarity(run_tag_sets[i], run_tag_sets[j])
                pairs.append(j_score)

        avg_jaccard = np.mean(pairs) if pairs else 1.0

        # Core tags (in ALL runs) vs variable tags (not in all runs)
        core_tags = run_tag_sets[0]
        all_tags = run_tag_sets[0]
        for rt in run_tag_sets[1:]:
            core_tags = core_tags & rt
            all_tags = all_tags | rt
        variable_tags = all_tags - core_tags

        results.append({
            'ProblemID': pid,
            'ScorePct': score_pct,
            'Condition': condition_name,
            'Run1_Tags': sorted(run_tag_sets[0]),
            'Run2_Tags': sorted(run_tag_sets[1]),
            'Run3_Tags': sorted(run_tag_sets[2]),
            'Core_Tags': sorted(core_tags),
            'Variable_Tags': sorted(variable_tags),
            'Num_Core': len(core_tags),
            'Num_Variable': len(variable_tags),
            'Avg_Jaccard': round(avg_jaccard, 4),
        })

        stability = "HIGH" if avg_jaccard >= 0.8 else ("MODERATE" if avg_jaccard >= 0.5 else "LOW")
        print(f"  {condition_name} Jaccard={avg_jaccard:.3f} [{stability}] Core={sorted(core_tags)} Variable={sorted(variable_tags)}")

consistency_df = pd.DataFrame(results)
print(f"\nTotal test cases: {len(consistency_df)}")


--- Problem 108 (Score: 68.4%) ---
  Baseline Run 1: ['ArrayIndex', 'If/Else', 'NestedFor'] (29.925s)
  Baseline Run 2: ['ArrayIndex', 'LogicBoolean', 'NestedFor'] (20.042s)
  Baseline Run 3: ['ArrayIndex', 'LogicBoolean', 'NestedFor'] (27.345s)
  Baseline Jaccard=0.667 [MODERATE] Core=['ArrayIndex', 'NestedFor'] Variable=['If/Else', 'LogicBoolean']
  Enriched Run 1: ['ArrayIndex', 'For', 'If/Else', 'NestedFor', 'While'] (49.484s)
  Enriched Run 2: ['ArrayIndex', 'For', 'NestedFor', 'While'] (29.999s)
  Enriched Run 3: ['ArrayIndex', 'If/Else', 'LogicAndNotOr', 'NestedFor'] (33.91s)
  Enriched Jaccard=0.544 [MODERATE] Core=['ArrayIndex', 'NestedFor'] Variable=['For', 'If/Else', 'LogicAndNotOr', 'While']

--- Problem 32 (Score: 54.5%) ---
  Baseline Run 1: ['For', 'If/Else', 'StringEqual', 'StringIndex'] (13.796s)
  Baseline Run 2: ['For', 'If/Else', 'StringIndex'] (12.086s)
  Baseline Run 3: ['For', 'If/Else', 'StringIndex', 'While'] (19.802s)
  Baseline Jaccard=0.700 [MODERATE] Core=

In [7]:
print("=" * 70)
print("CONSISTENCY ANALYSIS")
print("=" * 70)

# Overall
overall_jaccard = consistency_df['Avg_Jaccard'].mean()
stability = "HIGH" if overall_jaccard >= 0.8 else ("MODERATE" if overall_jaccard >= 0.5 else "LOW")
print(f"\nOverall Average Jaccard Similarity: {overall_jaccard:.3f} [{stability}]")

# By condition
print(f"\nBy Condition:")
for condition in ["Baseline", "Enriched"]:
    cond_df = consistency_df[consistency_df['Condition'] == condition]
    avg_j = cond_df['Avg_Jaccard'].mean()
    avg_core = cond_df['Num_Core'].mean()
    avg_var = cond_df['Num_Variable'].mean()
    print(f"  {condition}: Jaccard={avg_j:.3f}, Avg Core Tags={avg_core:.1f}, Avg Variable Tags={avg_var:.1f}")

# Per-problem detail
print(f"\nPer-Problem Detail:")
print(f"{'Problem':>8} {'Score':>7} {'Condition':<10} {'Jaccard':>8} {'Core':>5} {'Variable':>8} {'Core Tags'}")
print("-" * 85)
for _, r in consistency_df.iterrows():
    core_str = ", ".join(r['Core_Tags']) if r['Core_Tags'] else "(none)"
    var_str = ", ".join(r['Variable_Tags']) if r['Variable_Tags'] else "(none)"
    print(f"{r['ProblemID']:>8} {r['ScorePct']:>6.1f}% {r['Condition']:<10} {r['Avg_Jaccard']:>8.3f} {r['Num_Core']:>5} {r['Num_Variable']:>8}   {core_str}")
    if r['Variable_Tags']:
        print(f"{'':>8} {'':>7} {'':>10} {'':>8} {'':>5} {'':>8}   Variable: {var_str}")

# Verdict
print(f"\n{'=' * 70}")
print(f"VERDICT")
print(f"{'=' * 70}")
if overall_jaccard >= 0.8:
    print("HIGH consistency. Single-run results are trustworthy.")
    print("No need for multiple runs or majority voting.")
elif overall_jaccard >= 0.5:
    print("MODERATE consistency. Core tags are stable but some variation exists.")
    print("Single-run results capture the main signal. Variable tags add noise")
    print("but do not invalidate the core findings.")
    print("Recommendation: Report core tags as primary findings, note variability as limitation.")
else:
    print("LOW consistency. Results vary significantly between runs.")
    print("Recommendation: Run each condition 3x and use majority voting before reporting.")

CONSISTENCY ANALYSIS

Overall Average Jaccard Similarity: 0.614 [MODERATE]

By Condition:
  Baseline: Jaccard=0.650, Avg Core Tags=2.6, Avg Variable Tags=2.6
  Enriched: Jaccard=0.577, Avg Core Tags=2.6, Avg Variable Tags=3.8

Per-Problem Detail:
 Problem   Score Condition   Jaccard  Core Variable Core Tags
-------------------------------------------------------------------------------------
     108   68.4% Baseline      0.667     2        2   ArrayIndex, NestedFor
                                                      Variable: If/Else, LogicBoolean
     108   68.4% Enriched      0.544     2        4   ArrayIndex, NestedFor
                                                      Variable: For, If/Else, LogicAndNotOr, While
      32   54.5% Baseline      0.700     3        2   For, If/Else, StringIndex
                                                      Variable: StringEqual, While
      32   54.5% Enriched      0.733     3        2   For, If/Else, StringIndex
                         

In [8]:
from datetime import datetime

results_dir = ROOT / 'results' / '08_consistency_test'
results_dir.mkdir(parents=True, exist_ok=True)

consistency_df.to_csv(results_dir / 'consistency_results.csv', index=False)

metadata = {
    "experiment": "08_consistency_test",
    "run_timestamp": datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "student_id": TARGET_STUDENT_ID,
    "num_runs": NUM_RUNS,
    "problems_tested": FAILING_PROBLEM_IDS,
    "conditions": ["Baseline", "Enriched"],
    "overall_jaccard": round(float(overall_jaccard), 4),
    "verdict": stability,
    "baseline_avg_jaccard": round(float(consistency_df[consistency_df['Condition'] == 'Baseline']['Avg_Jaccard'].mean()), 4),
    "enriched_avg_jaccard": round(float(consistency_df[consistency_df['Condition'] == 'Enriched']['Avg_Jaccard'].mean()), 4),
}

with open(results_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to: {results_dir}")
print(f"  consistency_results.csv")
print(f"  metadata.json")

Saved to: /mnt/d/Projects/kintsugi/results/08_consistency_test
  consistency_results.csv
  metadata.json
